=====================================
# Step 1: Import Required Libraries
=====================================

Import all the necessary Python libraries required for data manipulation, visualization, and date-time operations.

In [21]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
from zoneinfo import ZoneInfo

=====================================
# Step 2: Load Both Datasets
=====================================

Load the Google Play Store dataset and the User Reviews dataset into Pandas DataFrames.

In [22]:
play_store_data = pd.read_csv(
    r"C:\Users\Vansh Sharma\Downloads\Play Store Data (1).csv"
)

play_store_data.head()
 
review = pd.read_csv(
    r"C:\Users\Vansh Sharma\Downloads\User Reviews (1).csv"
)

review.head()

,App,Translated_Review,Sentiment,Sentiment_Polarity,Sentiment_Subjectivity
0,10 Best Foods for You,I like eat delicious food. That's I'm cooking ...,Positive,1.00,0.533333
1,10 Best Foods for You,This help eating healthy exercise regular basis,Positive,0.25,0.288462
2,10 Best Foods for You,NaN,NaN,NaN,NaN
3,10 Best Foods for You,Works great especially going grocery store,Positive,0.40,0.875000
4,10 Best Foods for You,Best idea us,Positive,1.00,0.300000


=====================================
# Step 3: Data Cleaning - Play Store Dataset
=====================================

Clean the Play Store dataset by removing duplicate records, converting data types, handling missing values, and preparing the Size, Reviews, Installs, Price, and Rating columns for analysis.

In [23]:
# Remove duplicate apps
play_store_data = play_store_data.drop_duplicates(subset='App')

# Convert Rating to numeric
play_store_data['Rating'] = pd.to_numeric(play_store_data['Rating'], errors='coerce')

# Convert Reviews to numeric
play_store_data['Reviews'] = pd.to_numeric(play_store_data['Reviews'], errors='coerce')

# Clean Installs column
play_store_data['Installs'] = (
    play_store_data['Installs']
    .astype(str)
    .str.replace(',', '', regex=False)
    .str.replace('+', '', regex=False)
)

play_store_data['Installs'] = pd.to_numeric(play_store_data['Installs'], errors='coerce')

# Clean Price column
play_store_data['Price'] = (
    play_store_data['Price']
    .astype(str)
    .str.replace('$', '', regex=False)
)

play_store_data['Price'] = pd.to_numeric(play_store_data['Price'], errors='coerce')

# Convert Size to MB
def convert_size(size):
    if pd.isna(size):
        return np.nan

    size = str(size)

    if size.endswith('M'):
        return float(size[:-1])

    elif size.endswith('k'):
        return float(size[:-1]) / 1024

    else:
        return np.nan

play_store_data['Size_MB'] = play_store_data['Size'].apply(convert_size)

# Remove rows with missing important values
play_store_data = play_store_data.dropna(
    subset=['Rating', 'Reviews', 'Installs', 'Size_MB']
)

print(play_store_data.shape)
play_store_data.head()

(7027, 14)


,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver,Size_MB
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,4.1,159.0,19M,10000.0,Free,0.0,Everyone,Art & Design,"January 7, 2018",1.0.0,4.0.3 and up,19.0
1,Coloring book moana,ART_AND_DESIGN,3.9,967.0,14M,500000.0,Free,0.0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up,14.0
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,4.7,87510.0,8.7M,5000000.0,Free,0.0,Everyone,Art & Design,"August 1, 2018",1.2.4,4.0.3 and up,8.7
3,Sketch - Draw & Paint,ART_AND_DESIGN,4.5,215644.0,25M,50000000.0,Free,0.0,Teen,Art & Design,"June 8, 2018",Varies with device,4.2 and up,25.0
4,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,4.3,967.0,2.8M,100000.0,Free,0.0,Everyone,Art & Design;Creativity,"June 20, 2018",1.1,4.4 and up,2.8


=====================================
# Step 4: Data Cleaning - User Reviews Dataset
=====================================

Prepare the User Reviews dataset by converting the Sentiment Subjectivity column to numeric format, removing missing values, and aggregating duplicate app reviews.

In [24]:
# Keep required columns
review = review[['App', 'Sentiment_Subjectivity']]

# Convert Sentiment_Subjectivity to numeric
review['Sentiment_Subjectivity'] = pd.to_numeric(
    review['Sentiment_Subjectivity'],
    errors='coerce'
)

# Remove missing values
review = review.dropna(subset=['Sentiment_Subjectivity'])

# Remove duplicate apps by taking mean subjectivity
review = (
    review.groupby('App', as_index=False)
    ['Sentiment_Subjectivity']
    .mean()
)

print(review.shape)
review.head()

(865, 2)


,App,Sentiment_Subjectivity
0,10 Best Foods for You,0.495455
1,104 找工作 - 找工作 找打工 找兼職 履歷健檢 履歷診療室,0.545516
2,11st,0.443957
3,1800 Contacts - Lens Store,0.591098
4,1LINE – One Line with One Touch,0.557315


=====================================
# Step 5: Merge Both Datasets
=====================================

Merge the Play Store dataset with the User Reviews dataset using the App column to include Sentiment Subjectivity for each application.

In [25]:
merged_data = pd.merge(
    play_store_data,
    review,
    on='App',
    how='inner'
)

print(merged_data.shape)
merged_data.head()

(568, 15)


,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver,Size_MB,Sentiment_Subjectivity
0,Coloring book moana,ART_AND_DESIGN,3.9,967.0,14M,500000.0,Free,0.0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up,14.0,0.641540
1,Garden Coloring Book,ART_AND_DESIGN,4.4,13791.0,33M,1000000.0,Free,0.0,Everyone,Art & Design,"September 20, 2017",2.9.2,3.0 and up,33.0,0.523447
2,FlipaClip - Cartoon animation,ART_AND_DESIGN,4.3,194216.0,39M,5000000.0,Free,0.0,Everyone,Art & Design,"August 3, 2018",2.2.5,4.0.3 and up,39.0,0.679226
3,Boys Photo Editor - Six Pack & Men's Suit,ART_AND_DESIGN,4.1,654.0,12M,100000.0,Free,0.0,Everyone,Art & Design,"March 20, 2018",1.1,4.0.3 and up,12.0,0.479298
4,Colorfit - Drawing & Coloring,ART_AND_DESIGN,4.7,20260.0,25M,500000.0,Free,0.0,Everyone,Art & Design;Creativity,"October 11, 2017",1.0.8,4.0.3 and up,25.0,0.572762


=====================================
# Step 6: Apply Required Filters
=====================================

Filter the merged dataset according to the specified business requirements:

- Rating > 3.5
- Reviews > 500
- Installs > 50,000
- Sentiment Subjectivity > 0.5
- App name should not contain the letter 'S' or 's'
- Include only the specified categories

In [26]:
# Required Categories
required_categories = [
    'GAME',
    'BEAUTY',
    'BUSINESS',
    'COMICS',
    'COMMUNICATION',
    'DATING',
    'ENTERTAINMENT',
    'SOCIAL',
    'EVENTS'
]

# Apply all filters
filtered_data = merged_data[
    (merged_data['Rating'] > 3.5) &
    (merged_data['Reviews'] > 500) &
    (merged_data['Installs'] > 50000) &
    (merged_data['Sentiment_Subjectivity'] > 0.5) &
    (~merged_data['App'].str.contains('S', case=False, na=False)) &
    (merged_data['Category'].isin(required_categories))
].copy()

print("Filtered Dataset Shape:", filtered_data.shape)
filtered_data.head()

Filtered Dataset Shape: (23, 15)


,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver,Size_MB,Sentiment_Subjectivity
30,Google Primer,BUSINESS,4.4,62272.0,18M,10000000.0,Free,0.0,Everyone,Business,"June 26, 2018",3.550.2,4.1 and up,18.000000,0.675000
31,Call Blocker,BUSINESS,4.6,188841.0,3.2M,5000000.0,Free,0.0,Everyone,Business,"June 21, 2018",1.1.13,4.0 and up,3.200000,0.655431
54,"CallApp: Caller ID, Blocker & Phone Call Recorder",COMMUNICATION,4.4,483565.0,20M,10000000.0,Free,0.0,Everyone,Communication,"July 29, 2018",1.286,4.1 and up,20.000000,0.506481
56,Caller ID +,COMMUNICATION,4.0,9498.0,118k,1000000.0,Free,0.0,Everyone,Communication,"June 7, 2016",5.28.0,2.3 and up,0.115234,0.600000
59,"Hily: Dating, Chat, Match, Meet & Hook up",DATING,4.1,2556.0,56M,100000.0,Free,0.0,Mature 17+,Dating,"August 1, 2018",2.5.2,4.1 and up,56.000000,0.555331


=====================================
# Step 7: Translate Category Names
=====================================

Translate selected category names for visualization purposes only.

- Beauty → सौंदर्य (Hindi)
- Business → வணிகம் (Tamil)
- Dating → Partnersuche (German)

The original dataset remains unchanged.

In [27]:
# Create translated category names for display
category_translation = {
    'BEAUTY': 'सौंदर्य',          # Hindi
    'BUSINESS': 'வணிகம்',        # Tamil
    'DATING': 'Partnersuche'     # German
}

filtered_data['Category_Display'] = filtered_data['Category'].replace(category_translation)

filtered_data[['Category', 'Category_Display']].drop_duplicates()

,Category,Category_Display
30,BUSINESS,வணிகம்
54,COMMUNICATION,COMMUNICATION
59,DATING,Partnersuche
105,ENTERTAINMENT,ENTERTAINMENT
237,GAME,GAME
363,SOCIAL,SOCIAL


=====================================
# Step 8: Verify Time Constraint and bublle chart
=====================================

Check the current Indian Standard Time (IST). The Bubble Chart will only be displayed if the current time falls between 5:00 PM IST and 7:00 PM IST.

In [28]:
# Get current time in IST
current_time = datetime.now(ZoneInfo("Asia/Kolkata")).time()

# Allowed time: 5 PM to 7 PM IST
start_time = datetime.strptime("17:00", "%H:%M").time()
end_time = datetime.strptime("19:00", "%H:%M").time()

if start_time <= current_time <= end_time:

    plt.figure(figsize=(14,8))

    # Colors for each category
    color_map = {
        'GAME': 'pink',
        'सौंदर्य': 'blue',
        'வணிகம்': 'green',
        'COMICS': 'orange',
        'COMMUNICATION': 'purple',
        'Partnersuche': 'red',
        'ENTERTAINMENT': 'brown',
        'SOCIAL': 'cyan',
        'EVENTS': 'gray'
    }

    # Plot category-wise
    for category in filtered_data['Category_Display'].unique():

        category_data = filtered_data[
            filtered_data['Category_Display'] == category
        ]

        # Game category should always be pink
        original_category = category_data['Category'].iloc[0]

        color = "pink" if original_category == "GAME" else color_map.get(category, "black")

        plt.scatter(
            category_data['Size_MB'],
            category_data['Rating'],
            s=category_data['Installs'] / 1000,
            c=color,
            alpha=0.6,
            edgecolors='black',
            label=category
        )

    plt.title("Bubble Chart: App Size vs Rating", fontsize=16)
    plt.xlabel("App Size (MB)", fontsize=12)
    plt.ylabel("Average Rating", fontsize=12)
    plt.legend(title="Category")
    plt.grid(True)
    plt.tight_layout()
    plt.show()

else:
    print("Bubble chart is available only between 5 PM IST and 7 PM IST.")

Bubble chart is available only between 5 PM IST and 7 PM IST.
